# Notebook 02 — Prédiction du tour de pitstop (Supervisé)

**Objectif** : Prédire si un arrêt au stand va avoir lieu à chaque tour (classification binaire)  
**Features** : âge pneus, compound, position, météo, stint, temps au tour  
**Modèles** : Random Forest + XGBoost avec GridSearchCV  
**Évaluation** : accuracy, F1-score, matrice de confusion, feature importance


In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from src.data_loader import load_session, load_multiple_seasons, get_laps_features, get_weather
from src.supervised import (
    build_dataset,
    prepare_Xy,
    _temporal_split,
    train_random_forest,
    train_xgboost,
    evaluate_model,
    plot_confusion_matrix,
    plot_feature_importance,
    plot_model_comparison,
    HAS_XGB,
)

plt.rcParams['figure.dpi'] = 120
print(f'XGBoost disponible : {HAS_XGB}')

## 1. Chargement des données

In [ ]:
# Option A : une seule session (2025 avec fallback 2024)
YEAR, GP_NAME, SESSION_TYPE = 2025, 'Bahrain', 'R'
session = load_session(YEAR, GP_NAME, SESSION_TYPE)
if session is None:
    print(f'  [INFO] {YEAR} {GP_NAME} indisponible — fallback 2024')
    YEAR = 2024
    session = load_session(YEAR, GP_NAME, SESSION_TYPE)
sessions = [session] if session else []

# Option B : multi-saisons (décommenter pour plus de données)
# sessions = load_multiple_seasons(start=2021, end=2024, session_type='R')

print(f'{len(sessions)} session(s) chargée(s)')

In [ ]:
df = build_dataset(sessions)
print(f'Dataset : {len(df)} tours')
print(f'Pitstops : {df["HasPitStop"].sum()} ({100*df["HasPitStop"].mean():.1f}%)')
df.head()

## 2. Exploration & équilibre des classes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Équilibre des classes
counts = df['HasPitStop'].value_counts()
axes[0].bar(['Pas de pit', 'Pit Stop'], counts.values, color=['steelblue', '#e74c3c'], alpha=0.8)
axes[0].set_title('Équilibre des classes')
axes[0].set_ylabel('Nombre de tours')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 1, str(v), ha='center')

# TyreLife par pitstop
for label, grp in df.groupby('HasPitStop'):
    axes[1].hist(grp['TyreLife'].dropna(), bins=30, alpha=0.6,
                 label='Pit' if label else 'No Pit', density=True)
axes[1].set_title('Distribution TyreLife')
axes[1].legend()

# Position par pitstop
for label, grp in df.groupby('HasPitStop'):
    axes[2].hist(grp['Position'].dropna(), bins=20, alpha=0.6,
                 label='Pit' if label else 'No Pit', density=True)
axes[2].set_title('Distribution Position')
axes[2].legend()

plt.suptitle('Exploration du dataset supervisé', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Taux de pitstop par compound
if 'Compound' in df.columns:
    pit_by_compound = df.groupby('Compound')['HasPitStop'].agg(['mean', 'count'])
    pit_by_compound.columns = ['Taux pitstop', 'Nombre de tours']
    pit_by_compound['Taux pitstop'] = (pit_by_compound['Taux pitstop'] * 100).round(1)
    display(pit_by_compound)

## 3. Préparation du dataset (features / cible)

In [ ]:
X, y = prepare_Xy(df)
# Split temporel pour éviter le data leakage : les derniers tours (ou sessions) servent de test
X_train, X_test, y_train, y_test = _temporal_split(df, X, y, test_size=0.2)
print(f'Split temporel — Train : {len(X_train)}  |  Test : {len(X_test)}')
print(f'Features : {X.columns.tolist()}')

## 4. Entraînement — Random Forest

In [ ]:
rf_model, rf_params = train_random_forest(X_train, y_train)
print(f'Meilleurs paramètres RF : {rf_params}')

In [ ]:
metrics_rf = evaluate_model(rf_model, X_test, y_test, 'Random Forest')

In [ ]:
fig = plot_confusion_matrix(y_test, metrics_rf['y_pred'], 'Random Forest')
plt.show()

In [ ]:
fig = plot_feature_importance(rf_model, X.columns.tolist(), 'Random Forest')
plt.show()

## 5. Entraînement — XGBoost

In [ ]:
metrics_xgb = None
xgb_model = None

if HAS_XGB:
    xgb_model, xgb_params = train_xgboost(X_train, y_train)
    print(f'Meilleurs paramètres XGB : {xgb_params}')
    if xgb_model:
        metrics_xgb = evaluate_model(xgb_model, X_test, y_test, 'XGBoost')
        
        fig = plot_confusion_matrix(y_test, metrics_xgb['y_pred'], 'XGBoost')
        plt.show()
        
        fig = plot_feature_importance(xgb_model, X.columns.tolist(), 'XGBoost')
        plt.show()
else:
    print('XGBoost non installé — pip install xgboost')

## 6. Comparaison des modèles

In [ ]:
fig = plot_model_comparison(metrics_rf, metrics_xgb)
plt.show()

print('\nRésumé :')
print(f'  Random Forest — Accuracy: {metrics_rf["accuracy"]:.4f}  F1: {metrics_rf["f1"]:.4f}')
if metrics_xgb:
    print(f'  XGBoost       — Accuracy: {metrics_xgb["accuracy"]:.4f}  F1: {metrics_xgb["f1"]:.4f}')

## 7. Analyse du seuil de probabilité

In [ ]:
from sklearn.metrics import precision_recall_curve, f1_score

proba_rf = rf_model.predict_proba(X_test)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, proba_rf)

# F1 pour chaque seuil
f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-8)
best_thresh = thresholds[f1_scores.argmax()]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, precisions[:-1], label='Précision', color='steelblue')
ax.plot(thresholds, recalls[:-1], label='Rappel', color='#e74c3c')
ax.plot(thresholds, f1_scores, label='F1-Score', color='#2ecc71', linestyle='--')
ax.axvline(best_thresh, color='gray', linestyle=':', label=f'Seuil optimal ({best_thresh:.2f})')
ax.set_xlabel('Seuil de probabilité')
ax.set_title('Précision / Rappel / F1 en fonction du seuil (RF)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Seuil optimal pour F1 : {best_thresh:.3f}')